# detfuse — Detector 評估

比較 **KeywordDetector**（L1 軟分數）、**Qwen2.5-0.5B**（L2 信心分數）與**並行融合**（α × L1 + (1-α) × L2）在偵測「免費食物」貼文的準確率。

執行環境：Colab（T4 GPU）

## 1. 安裝依賴

In [ ]:
!pip install -q transformers accelerate

## 2. KeywordDetector（直接複製 detector.py 的 L1 邏輯）

In [ ]:
import re

# 與 detector.py 保持一致
_FREE_WORDS = r'免費|free|請拿|拿走|多餘|多的|送人|不要了|剩食|剩菜|拿去|有需要|帶走|送出|分享'
_FOOD_WORDS = r'食物|食品|飯|麵|便當|零食|餅乾|水果|蔬菜|菜|湯|肉|蛋|麵包|吐司|料理|點心|糕|餅|粽|飲料|奶茶|咖啡|茶|寶特瓶|三明治|沙拉|漢堡|披薩|壽司|飯糰|泡麵|湯圓'
_PATTERN_FREE_FOOD    = re.compile(rf'(?=.*({_FREE_WORDS}))(?=.*({_FOOD_WORDS}))', re.IGNORECASE)
_PATTERN_NOT_FOOD     = re.compile(r'免費.*?(?:課程|諮詢|活動|講座|workshop|票|名額|參加|索取)', re.IGNORECASE)
_PATTERN_EVENT_FOOD   = re.compile(
    r'(?:研討會|活動|工作坊|演講|說明會|工作人員).{0,10}(?:便當|餐盒|餐點|飲料|食物|點心)',
    re.IGNORECASE,
)
_PATTERN_HAS_CATERING = re.compile(r'(?:有|免費|提供)供餐', re.IGNORECASE)


def keyword_detect(text: str) -> bool:
    if not text:
        return False
    if _PATTERN_NOT_FOOD.search(text):
        return False
    return bool(_PATTERN_FREE_FOOD.search(text))


def l1_score(text: str) -> float:
    """L1 軟分數，[0, 1]。"""
    score = 0.0
    if _PATTERN_NOT_FOOD.search(text):
        return 0.0
    if _PATTERN_FREE_FOOD.search(text):     score += 0.80
    if _PATTERN_EVENT_FOOD.search(text):    score += 0.60
    if _PATTERN_HAS_CATERING.search(text):  score += 0.50
    has_free = bool(re.search(_FREE_WORDS, text, re.IGNORECASE))
    has_food = bool(re.search(_FOOD_WORDS, text, re.IGNORECASE))
    if has_free and not _PATTERN_FREE_FOOD.search(text): score += 0.20
    if has_food and not _PATTERN_FREE_FOOD.search(text): score += 0.10
    return min(score, 1.0)


print('L1 patterns + l1_score() loaded')

## 3. 測試資料集

由 Claude Code 標記，來源：`data/categories/free_food/`

- **training_data.json**：894 筆（正例 212 / 負例 682）
- **test_data.json**：234 筆（正例 58 / 負例 176）

評估使用 `test_data.json`，欄位：`text`、`label`（1 = 免費食物，0 = 非）

In [ ]:
import json, urllib.request

BRANCH = 'feature/parallel-fusion'
BASE_URL = f'https://raw.githubusercontent.com/syoslyot/detfuse/{BRANCH}/data/categories/free_food'

def load_samples(filename):
    url = f'{BASE_URL}/{filename}'
    with urllib.request.urlopen(url) as r:
        data = json.loads(r.read())
    return [(d['label'], d['text']) for d in data]

TRAIN_SAMPLES = load_samples('training_data.json')
TEST_SAMPLES  = load_samples('test_data.json')
SAMPLES = TEST_SAMPLES  # 評估用 test set

print(f'training: {sum(l==1 for l,_ in TRAIN_SAMPLES)} 正例 / {sum(l==0 for l,_ in TRAIN_SAMPLES)} 負例')
print(f'test:     {sum(l==1 for l,_ in TEST_SAMPLES)} 正例 / {sum(l==0 for l,_ in TEST_SAMPLES)} 負例')

## 4. 評估 KeywordDetector

In [ ]:
def evaluate(name, predict_fn, samples):
    tp = fp = tn = fn = 0
    errors = []
    for label, text in samples:
        pred = predict_fn(text)
        if label == 1 and pred:     tp += 1
        elif label == 0 and not pred: tn += 1
        elif label == 0 and pred:
            fp += 1
            errors.append(('FP', text[:60]))
        else:
            fn += 1
            errors.append(('FN', text[:60]))

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall    = tp / (tp + fn) if (tp + fn) else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

    print(f'\n── {name} ──')
    print(f'  Precision: {precision:.2%}  Recall: {recall:.2%}  F1: {f1:.2%}')
    print(f'  TP={tp} FP={fp} TN={tn} FN={fn}')
    if errors:
        print('  錯誤案例:')
        for tag, t in errors:
            print(f'    [{tag}] {t}')
    return dict(name=name, precision=precision, recall=recall, f1=f1)

kw_result = evaluate('KeywordDetector (L1)', keyword_detect, SAMPLES)

## 5. Qwen2.5-0.5B-Instruct（L2 模型，需 GPU）

模擬 `OllamaDetector` 的邏輯，但改用 HuggingFace Transformers 直接跑。
這樣不需要 Ollama server，在 Colab 上也能驗證模型品質。

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
)
print(f'模型載入完成，裝置：{next(model.parameters()).device}')

In [ ]:
PROMPT_TMPL = (
    '判斷以下貼文是否在提供免費食物或飲料（可以現在就去拿）。\n'
    '請只輸出 0 到 9 的整數，代表信心程度（0 = 完全不是，9 = 完全確定是）。不要輸出其他任何文字。\n\n'
    '貼文：{text}'
)


def qwen_prob(text: str) -> float:
    """回傳 [0, 1] 信心分數（0-9 digit / 9.0）。"""
    messages = [
        {'role': 'system', 'content': '請只輸出 0 到 9 的整數。'},
        {'role': 'user', 'content': PROMPT_TMPL.format(text=text[:400])},
    ]
    ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    )
    if hasattr(ids, 'input_ids'):
        ids = ids.input_ids
    ids = ids.to(model.device)
    input_len = ids.shape[1]
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=5, do_sample=False)
    answer = tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip()
    for ch in answer:
        if ch.isdigit():
            return int(ch) / 9.0
    return 0.5  # 無法解析 → 中立


def qwen_detect(text: str) -> bool:
    return qwen_prob(text) > 0.5


# Smoke test
print(qwen_prob('有多的便當，免費拿走，在工程館'))   # expect ≥ 0.5
print(qwen_prob('出售二手書'))                        # expect < 0.5

In [ ]:
qwen_result = evaluate('Qwen2.5-0.5B (L2)', qwen_detect, SAMPLES)

## 6. 並行融合（α × L1 + (1-α) × L2）

L1 和 L2 各自計算分數，再加權融合——兩層都有發言權，都能糾正對方的誤判。

In [ ]:
def fuse(text: str, alpha: float = 0.35, threshold: float = 0.50) -> bool:
    s1 = l1_score(text)
    s2 = qwen_prob(text)
    return (alpha * s1 + (1 - alpha) * s2) > threshold


fusion_result = evaluate('Fusion α=0.35 τ=0.50', fuse, SAMPLES)

## 7. 比較結果

In [ ]:
print(f'\n{"模型":<28} {"Precision":>10} {"Recall":>10} {"F1":>10}')
print('-' * 62)
for r in [kw_result, qwen_result, fusion_result]:
    print(f"{r['name']:<28} {r['precision']:>10.2%} {r['recall']:>10.2%} {r['f1']:>10.2%}")